# Cross-Domain Generalization of Deep Learning Detectors for Rumors and Fake News in Online Social Networks

**Reproducible companion notebook.** Runs end to end on a free Google Colab GPU
(Runtime -> Change runtime type -> T4 GPU). No API keys, no manual downloads:
both datasets are pulled directly from public, directly-committed GitHub mirrors.

**What this notebook produces**
1. Two harmonized public corpora (LIAR short political fact-checks; McIntire full news articles).
2. Classical baselines (TF-IDF + Logistic Regression / Linear SVM / Multinomial NB).
3. A BiLSTM baseline.
4. Fine-tuned Transformer encoders (DistilBERT, RoBERTa).
5. **In-domain vs cross-domain generalization matrix** (the central experiment).
6. Calibration analysis (ECE, reliability diagrams, temperature scaling).
7. Model interpretability (linear coefficients; SHAP for the Transformer).
8. Statistical rigor: bootstrap 95% CIs and McNemar tests.

All outputs are written to `./results/` (a single `results.json` plus figures)
so the numbers reported in the paper are regenerated verbatim.

> Runtime on a free T4 GPU is approximately 15-25 minutes for the full run.


## 0. Environment and reproducibility

In [ ]:
# Install pinned dependencies (Colab already ships torch + CUDA)
!pip -q install "transformers>=4.44" "datasets>=2.20" shap scikit-learn statsmodels matplotlib 2>/dev/null
import os, random, json, re, time, sys
import numpy as np, torch
SEED = 42
os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
os.makedirs("results", exist_ok=True); os.makedirs("results/figs", exist_ok=True)
import transformers, sklearn
print("python", sys.version.split()[0], "| torch", torch.__version__,
      "| transformers", transformers.__version__, "| sklearn", sklearn.__version__,
      "| device", DEVICE)


## 1. Data acquisition (fully self-contained, no social-media API)

We deliberately avoid tweet-ID datasets (PHEME, Twitter15/16) whose text must be
rehydrated through the paywalled X API: such pipelines are no longer reproducible.
Instead we use two directly-committed public corpora and harmonize them to one
binary schema, `y = 1` (fake / misinformation) vs `y = 0` (real).

* **LIAR** (Wang, 2017): 12.8k short PolitiFact statements, six veracity labels.
  Binarized as {pants-fire, false, barely-true} -> fake; {half-true, mostly-true, true} -> real.
* **McIntire fake_or_real_news**: ~6.3k full-length news articles labeled FAKE/REAL.

The two differ sharply in length and register (tens vs thousands of characters),
which is what makes the cross-domain test informative.


In [ ]:
import pandas as pd, urllib.request, urllib.error, time
os.makedirs("data", exist_ok=True)
FILES = {
 "liar_train.tsv": "https://raw.githubusercontent.com/thiagorainmaker77/liar_dataset/master/train.tsv",
 "liar_valid.tsv": "https://raw.githubusercontent.com/thiagorainmaker77/liar_dataset/master/valid.tsv",
 "liar_test.tsv":  "https://raw.githubusercontent.com/thiagorainmaker77/liar_dataset/master/test.tsv",
 "mcintire.csv":   "https://raw.githubusercontent.com/lutzhamel/fake-news/master/data/fake_or_real_news.csv",
}
UA = {"User-Agent": "Mozilla/5.0 (reproducibility-notebook; academic research)"}
def fetch(url, path, retries=6):
    # exponential backoff handles GitHub raw HTTP 429 rate limiting
    for i in range(retries):
        try:
            req = urllib.request.Request(url, headers=UA)
            with urllib.request.urlopen(req, timeout=90) as r, open(path,"wb") as f:
                f.write(r.read())
            if os.path.getsize(path) > 0: return True
            raise IOError("empty file")
        except Exception as e:
            code = getattr(e, "code", None)
            if i < retries-1:
                wait = 5*(2**i); print(f"  {os.path.basename(path)}: {code or type(e).__name__}, retry in {wait}s"); time.sleep(wait); continue
            print(f"  {os.path.basename(path)}: FAILED ({code or e})"); return False
for fn, url in FILES.items():
    p = os.path.join("data", fn)
    if not os.path.exists(p) or os.path.getsize(p)==0: fetch(url, p)
    print(fn, os.path.getsize(p) if os.path.exists(p) else 0, "bytes")

# Fallback for LIAR via the datasets library if the raw mirror stays rate limited
if any(not os.path.exists(f"data/liar_{s}.tsv") or os.path.getsize(f"data/liar_{s}.tsv")==0
       for s in ["train","valid","test"]):
    print("Using Hugging Face datasets fallback for LIAR...")
    !pip -q install datasets 2>/dev/null
    from datasets import load_dataset
    dd = load_dataset("liar")
    lbl = {0:'false',1:'half-true',2:'mostly-true',3:'true',4:'barely-true',5:'pants-fire'}
    for hf_split, fn in [("train","train"),("validation","valid"),("test","test")]:
        df = dd[hf_split].to_pandas(); df["label"]=df["label"].map(lbl)
        df[["id","label","statement","subject","speaker","job_title","state_info","party_affiliation",
            "barely_true_counts","false_counts","half_true_counts","mostly_true_counts",
            "pants_on_fire_counts","context"]].to_csv(f"data/liar_{fn}.tsv", sep="\t", header=False, index=False)
    print("LIAR fallback written.")


In [ ]:
def clean(t):
    t = str(t).replace("\n"," ").replace("\r"," ")
    return re.sub(r"\s+"," ", t).strip()

# ---- LIAR ----
liar_cols = ['id','label','statement','subject','speaker','job','state','party',
             'barely_true','false','half_true','mostly_true','pants_fire','context']
FAKE_LIAR = {'pants-fire','false','barely-true'}
REAL_LIAR = {'half-true','mostly-true','true'}
def load_liar(split):
    df = pd.read_csv(f"data/liar_{split}.tsv", sep='\t', header=None, names=liar_cols)
    df = df[df['label'].isin(FAKE_LIAR|REAL_LIAR)].copy()
    df['text'] = df['statement'].map(clean)
    df['y'] = df['label'].isin(FAKE_LIAR).astype(int)
    return df[df['text'].str.len()>0][['text','y']].reset_index(drop=True)
liar = {s: load_liar(s) for s in ['train','valid','test']}

# ---- McIntire (stratified 80/10/10, seeded) ----
mc = pd.read_csv("data/mcintire.csv")
mc['text'] = (mc['title'].fillna('')+'. '+mc['text'].fillna('')).map(clean)
mc['y'] = (mc['label'].str.upper()=='FAKE').astype(int)
mc = mc[mc['text'].str.len()>0].sample(frac=1.0, random_state=SEED).reset_index(drop=True)
n=len(mc); a,b=int(0.8*n),int(0.9*n)
mcintire={'train':mc.iloc[:a][['text','y']].reset_index(drop=True),
          'valid':mc.iloc[a:b][['text','y']].reset_index(drop=True),
          'test': mc.iloc[b:][['text','y']].reset_index(drop=True)}

DS = {'LIAR': liar, 'McIntire': mcintire}
profile={}
for name,d in DS.items():
    profile[name]={'train_n':len(d['train']),'valid_n':len(d['valid']),'test_n':len(d['test']),
                   'train_fake_rate':round(float(d['train']['y'].mean()),4),
                   'mean_chars_train':int(d['train']['text'].str.len().mean()),
                   'median_chars_train':int(d['train']['text'].str.len().median())}
print(json.dumps(profile, indent=2))
RESULTS={'seed':SEED,'dataset_profile':profile}


## 2. Shared evaluation utilities

In [ ]:
from sklearn.metrics import (accuracy_score, precision_recall_fscore_support,
                             roc_auc_score, f1_score)
def evaluate(y_true,y_pred,y_score):
    p,r,f,_=precision_recall_fscore_support(y_true,y_pred,average=None,labels=[0,1],zero_division=0)
    pm,rm,fm,_=precision_recall_fscore_support(y_true,y_pred,average='macro',zero_division=0)
    try: auc=roc_auc_score(y_true,y_score)
    except Exception: auc=float('nan')
    return {'accuracy':round(accuracy_score(y_true,y_pred),4),'f1_macro':round(fm,4),
            'precision_macro':round(pm,4),'recall_macro':round(rm,4),
            'f1_fake':round(f[1],4),'recall_fake':round(r[1],4),'precision_fake':round(p[1],4),
            'roc_auc':round(float(auc),4)}

def ece(y_true,y_prob,bins=10):
    y_true=np.asarray(y_true); y_prob=np.asarray(y_prob)
    pred=(y_prob>=0.5).astype(int); conf=np.maximum(y_prob,1-y_prob)
    correct=(pred==y_true).astype(float); edges=np.linspace(0,1,bins+1); e=0.0; N=len(y_true)
    for i in range(bins):
        m=(conf>edges[i])&(conf<=edges[i+1]) if i>0 else (conf>=edges[i])&(conf<=edges[i+1])
        if m.sum()==0: continue
        e+=(m.sum()/N)*abs(correct[m].mean()-conf[m].mean())
    return round(float(e),4)

def bootstrap_ci(y_true,y_pred,n_boot=1000,seed=SEED):
    rng=np.random.default_rng(seed); y_true=np.asarray(y_true); y_pred=np.asarray(y_pred); N=len(y_true); v=[]
    for _ in range(n_boot):
        idx=rng.integers(0,N,N); v.append(f1_score(y_true[idx],y_pred[idx],pos_label=1,zero_division=0))
    return [round(float(np.percentile(v,2.5)),4), round(float(np.percentile(v,97.5)),4)]


## 3. Classical baselines (in-domain and cross-domain)

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.naive_bayes import MultinomialNB

def make_vec():
    return TfidfVectorizer(lowercase=True,stop_words='english',ngram_range=(1,2),
                           min_df=2,max_df=0.9,sublinear_tf=True,max_features=50000)
def scores(m,X):
    if hasattr(m,'predict_proba'): return m.predict_proba(X)[:,1]
    d=m.decision_function(X); return (d-d.min())/(d.max()-d.min()+1e-9)
MODELS={'LogReg':lambda:LogisticRegression(max_iter=2000,C=4.0,class_weight='balanced'),
        'LinSVM':lambda:LinearSVC(C=1.0,class_weight='balanced'),
        'MultiNB':lambda:MultinomialNB()}
fitted={}; RESULTS['classical_in_domain']={}
for dn,d in DS.items():
    vec=make_vec(); Xtr=vec.fit_transform(d['train']['text']); ytr=d['train']['y'].values
    Xte=vec.transform(d['test']['text']); yte=d['test']['y'].values
    RESULTS['classical_in_domain'][dn]={}
    for mn,mf in MODELS.items():
        clf=mf(); clf.fit(Xtr,ytr); yp=clf.predict(Xte)
        RESULTS['classical_in_domain'][dn][mn]=evaluate(yte,yp,scores(clf,Xte))
        fitted[(dn,mn)]=(vec,clf)
    print(dn, RESULTS['classical_in_domain'][dn]['LogReg'])


In [ ]:
# Cross-domain transfer (train on source, test on target) using LogReg
RESULTS['classical_cross_domain']={}; RESULTS['generalization_gap']={}
for src in DS:
    vec,clf=fitted[(src,'LogReg')]
    for tgt in DS:
        if src==tgt: continue
        Xte=vec.transform(DS[tgt]['test']['text']); yte=DS[tgt]['test']['y'].values
        RESULTS['classical_cross_domain'][f"{src}->{tgt}"]=evaluate(yte,clf.predict(Xte),scores(clf,Xte))
        gap=RESULTS['classical_in_domain'][tgt]['LogReg']['f1_macro']-RESULTS['classical_cross_domain'][f"{src}->{tgt}"]['f1_macro']
        RESULTS['generalization_gap'][f"{src}->{tgt}"]=round(gap,4)
print("cross-domain:", json.dumps(RESULTS['classical_cross_domain'],indent=2))
print("gap:", RESULTS['generalization_gap'])


## 4. BiLSTM baseline (PyTorch)

In [ ]:
from torch.utils.data import Dataset, DataLoader
from collections import Counter
import torch.nn as nn

def tokenize(t): return re.findall(r"[a-z']+", t.lower())
class Vocab:
    def __init__(self, texts, max_size=30000, min_freq=2):
        c=Counter(w for t in texts for w in tokenize(t))
        self.itos=['<pad>','<unk>']+[w for w,f in c.most_common(max_size) if f>=min_freq]
        self.stoi={w:i for i,w in enumerate(self.itos)}
    def enc(self,t,maxlen):
        ids=[self.stoi.get(w,1) for w in tokenize(t)][:maxlen]
        return ids+[0]*(maxlen-len(ids))
class TextDS(Dataset):
    def __init__(self,df,vocab,maxlen):
        self.x=[vocab.enc(t,maxlen) for t in df['text']]; self.y=df['y'].tolist()
    def __len__(self): return len(self.y)
    def __getitem__(self,i): return torch.tensor(self.x[i]), torch.tensor(self.y[i])
class BiLSTM(nn.Module):
    def __init__(self,V,emb=128,hid=128):
        super().__init__(); self.emb=nn.Embedding(V,emb,padding_idx=0)
        self.lstm=nn.LSTM(emb,hid,batch_first=True,bidirectional=True)
        self.drop=nn.Dropout(0.3); self.fc=nn.Linear(2*hid,2)
    def forward(self,x):
        e=self.emb(x); o,_=self.lstm(e); h=self.drop(o.mean(1)); return self.fc(h)

def run_bilstm(dn, maxlen, epochs=4):
    d=DS[dn]; vocab=Vocab(d['train']['text'])
    tr=DataLoader(TextDS(d['train'],vocab,maxlen),batch_size=64,shuffle=True)
    te=DataLoader(TextDS(d['test'],vocab,maxlen),batch_size=128)
    m=BiLSTM(len(vocab.itos)).to(DEVICE); opt=torch.optim.Adam(m.parameters(),lr=1e-3)
    lossf=nn.CrossEntropyLoss()
    for ep in range(epochs):
        m.train()
        for xb,yb in tr:
            xb,yb=xb.to(DEVICE),yb.to(DEVICE); opt.zero_grad()
            lossf(m(xb),yb).backward(); opt.step()
    m.eval(); ys=[]; yp=[]; yt=[]
    with torch.no_grad():
        for xb,yb in te:
            p=torch.softmax(m(xb.to(DEVICE)),1)[:,1].cpu().numpy()
            ys+=p.tolist(); yp+=(p>=0.5).astype(int).tolist(); yt+=yb.tolist()
    return evaluate(yt,yp,ys), (m,vocab)
RESULTS['bilstm_in_domain']={}
MAXLEN={'LIAR':40,'McIntire':400}
bilstm_fitted={}
for dn in DS:
    res,mv=run_bilstm(dn,MAXLEN[dn]); RESULTS['bilstm_in_domain'][dn]=res; bilstm_fitted[dn]=(mv,MAXLEN[dn])
    print(dn,res)


In [ ]:
# BiLSTM cross-domain
RESULTS['bilstm_cross_domain']={}
for src in DS:
    (m,vocab),maxlen=bilstm_fitted[src]; m.eval()
    for tgt in DS:
        if src==tgt: continue
        te=DataLoader(TextDS(DS[tgt]['test'],vocab,maxlen),batch_size=128)
        ys=[];yp=[];yt=[]
        with torch.no_grad():
            for xb,yb in te:
                p=torch.softmax(m(xb.to(DEVICE)),1)[:,1].cpu().numpy()
                ys+=p.tolist();yp+=(p>=0.5).astype(int).tolist();yt+=yb.tolist()
        RESULTS['bilstm_cross_domain'][f"{src}->{tgt}"]=evaluate(yt,yp,ys)
print(json.dumps(RESULTS['bilstm_cross_domain'],indent=2))


## 5. Transformer encoders (DistilBERT, RoBERTa)

Fine-tuned with the Hugging Face `Trainer`. LIAR uses `max_len=64` (statements are
short); McIntire uses `max_len=256` (leading tokens of each article). Three epochs,
learning rate 2e-5, batch size 16, mixed precision on GPU, seed 42. We store the
per-example probabilities so the same predictions feed calibration and SHAP.


In [ ]:
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                          TrainingArguments, Trainer, set_seed)
import torch
set_seed(SEED)
TRANSFORMERS = ["distilbert-base-uncased", "roberta-base"]
TMAXLEN = {'LIAR':64, 'McIntire':256}
EPOCHS = 3

class HFDataset(torch.utils.data.Dataset):
    def __init__(self, df, tok, maxlen):
        self.enc = tok(list(df['text']), truncation=True, max_length=maxlen, padding='max_length')
        self.y = df['y'].tolist()
    def __len__(self): return len(self.y)
    def __getitem__(self, i):
        item = {k: torch.tensor(v[i]) for k,v in self.enc.items()}
        item['labels'] = torch.tensor(self.y[i]); return item

def predict_probs(trainer, ds):
    logits = trainer.predict(ds).predictions
    return torch.softmax(torch.tensor(logits), dim=1)[:,1].numpy()

transformer_fitted = {}          # (model, dataset) -> (trainer, tokenizer, maxlen)
transformer_test_probs = {}      # (model, dataset) -> probs on that dataset's own test
RESULTS['transformer_in_domain'] = {}
for mname in TRANSFORMERS:
    RESULTS['transformer_in_domain'][mname] = {}
    for dn, d in DS.items():
        tok = AutoTokenizer.from_pretrained(mname)
        model = AutoModelForSequenceClassification.from_pretrained(mname, num_labels=2)
        ml = TMAXLEN[dn]
        args = TrainingArguments(output_dir=f"ckpt/{mname.split('/')[-1]}_{dn}",
                    num_train_epochs=EPOCHS, per_device_train_batch_size=16,
                    per_device_eval_batch_size=64, learning_rate=2e-5, weight_decay=0.01,
                    warmup_ratio=0.06, fp16=torch.cuda.is_available(), seed=SEED,
                    logging_steps=200, report_to=[], eval_strategy="no", save_strategy="no")
        trainer = Trainer(model=model, args=args,
                          train_dataset=HFDataset(d['train'], tok, ml))
        trainer.train()
        te = HFDataset(d['test'], tok, ml)
        probs = predict_probs(trainer, te); yte = d['test']['y'].values
        RESULTS['transformer_in_domain'][mname][dn] = evaluate(yte, (probs>=0.5).astype(int), probs)
        transformer_fitted[(mname,dn)] = (trainer, tok, ml)
        transformer_test_probs[(mname,dn)] = (yte, probs)
        print(mname, dn, RESULTS['transformer_in_domain'][mname][dn])


In [ ]:
# Transformer cross-domain transfer
RESULTS['transformer_cross_domain'] = {}
for mname in TRANSFORMERS:
    RESULTS['transformer_cross_domain'][mname] = {}
    for src in DS:
        trainer, tok, ml = transformer_fitted[(mname, src)]
        for tgt in DS:
            if src == tgt: continue
            te = HFDataset(DS[tgt]['test'], tok, TMAXLEN[src])
            probs = predict_probs(trainer, te); yte = DS[tgt]['test']['y'].values
            RESULTS['transformer_cross_domain'][mname][f"{src}->{tgt}"] = \
                evaluate(yte, (probs>=0.5).astype(int), probs)
    print(mname, json.dumps(RESULTS['transformer_cross_domain'][mname], indent=2))


## 6. Calibration (ECE, reliability diagrams, temperature scaling)

In [ ]:
import matplotlib.pyplot as plt
RESULTS['calibration_ece']={}
# classical LogReg in-domain + cross-domain
for dn,d in DS.items():
    vec,lr=fitted[(dn,'LogReg')]; Xte=vec.transform(d['test']['text'])
    RESULTS['calibration_ece'][f"LogReg {dn} in-domain"]=ece(d['test']['y'].values, lr.predict_proba(Xte)[:,1])
for src in DS:
    for tgt in DS:
        if src==tgt: continue
        vec,lr=fitted[(src,'LogReg')]; Xte=vec.transform(DS[tgt]['test']['text'])
        RESULTS['calibration_ece'][f"LogReg {src}->{tgt}"]=ece(DS[tgt]['test']['y'].values, lr.predict_proba(Xte)[:,1])
# transformer in-domain (primary model = first in list)
pm=TRANSFORMERS[0]
for dn in DS:
    yte,probs=transformer_test_probs[(pm,dn)]
    RESULTS['calibration_ece'][f"{pm} {dn} in-domain"]=ece(yte,probs)
print(json.dumps(RESULTS['calibration_ece'],indent=2))

def reliability(y,p,title,path):
    edges=np.linspace(0,1,11); conf=np.maximum(p,1-p); pred=(p>=0.5).astype(int); corr=(pred==y).astype(float)
    xs=[];ys=[]
    for i in range(10):
        m=(conf>edges[i])&(conf<=edges[i+1]) if i>0 else (conf>=edges[i])&(conf<=edges[i+1])
        if m.sum()>0: xs.append(conf[m].mean()); ys.append(corr[m].mean())
    plt.figure(figsize=(4,4)); plt.plot([0.5,1],[0.5,1],'--',color='gray')
    plt.plot(xs,ys,'o-'); plt.xlabel('confidence'); plt.ylabel('accuracy'); plt.title(title)
    plt.tight_layout(); plt.savefig(path,dpi=150); plt.close()
yte,probs=transformer_test_probs[(pm,'McIntire')]; reliability(yte,probs,f"{pm} McIntire in-domain","results/figs/reliability_indomain.png")
print("saved reliability diagram")


## 7. Interpretability (linear coefficients + SHAP on the Transformer)

In [ ]:
# Linear coefficient evidence: what tokens drive each dataset's classifier?
RESULTS['top_features']={}
for dn in DS:
    vec,lr=fitted[(dn,'LogReg')]; names=np.array(vec.get_feature_names_out()); c=lr.coef_[0]
    RESULTS['top_features'][dn]={'fake_indicative':names[np.argsort(c)[-15:]][::-1].tolist(),
                                 'real_indicative':names[np.argsort(c)[:15]].tolist()}
print(json.dumps(RESULTS['top_features'],indent=2))
# Vocabulary overlap quantifies domain distance
from itertools import combinations
RESULTS['vocab_jaccard_top2000']={}
def tv(dn,k=2000):
    v,_=fitted[(dn,'LogReg')]; return set(v.get_feature_names_out()[:k])
for a,b in combinations(list(DS),2):
    RESULTS['vocab_jaccard_top2000'][f"{a}|{b}"]=round(len(tv(a)&tv(b))/len(tv(a)|tv(b)),4)
print(RESULTS['vocab_jaccard_top2000'])


In [ ]:
# SHAP on the fine-tuned Transformer (token attributions on sample test items)
import shap
pm=TRANSFORMERS[0]; trainer,tok,ml=transformer_fitted[(pm,'McIntire')]
mdl=trainer.model.eval()
def f(texts):
    enc=tok(list(texts),truncation=True,max_length=ml,padding=True,return_tensors='pt').to(DEVICE)
    with torch.no_grad(): return torch.softmax(mdl(**enc).logits,1).cpu().numpy()
masker=shap.maskers.Text(tok)
explainer=shap.Explainer(f, masker, output_names=['REAL','FAKE'])
sample=DS['McIntire']['test']['text'].iloc[:6].tolist()
sv=explainer(sample)
# store top contributing tokens for the FAKE class of the first sample
vals=sv[0,:,1].values; toks=sv[0,:,1].data
order=np.argsort(np.abs(vals))[::-1][:12]
RESULTS['shap_example']={'text_preview':sample[0][:160],
    'top_tokens':[[str(toks[i]).strip(), round(float(vals[i]),4)] for i in order]}
print(json.dumps(RESULTS['shap_example'],indent=2))
try:
    shap.plots.text(sv[0], display=False)
except Exception as e:
    print("interactive SHAP text plot is best viewed inline:", e)


## 8. Statistical rigor (bootstrap CIs, McNemar)

In [ ]:
from statsmodels.stats.contingency_tables import mcnemar
RESULTS['bootstrap_ci_f1fake']={}; RESULTS['mcnemar_logreg_vs_svm']={}
for dn,d in DS.items():
    vec,lr=fitted[(dn,'LogReg')]; _,sv=fitted[(dn,'LinSVM')]
    Xte=vec.transform(d['test']['text']); yte=d['test']['y'].values
    lp=lr.predict(Xte); sp=sv.predict(Xte)
    RESULTS['bootstrap_ci_f1fake'][dn]=bootstrap_ci(yte,lp)
    tb=[[int(((lp==yte)&(sp==yte)).sum()),int(((lp==yte)&(sp!=yte)).sum())],
        [int(((lp!=yte)&(sp==yte)).sum()),int(((lp!=yte)&(sp!=yte)).sum())]]
    r=mcnemar(tb,exact=False,correction=True)
    RESULTS['mcnemar_logreg_vs_svm'][dn]={'statistic':round(float(r.statistic),4),'pvalue':round(float(r.pvalue),6)}
print("CI:",RESULTS['bootstrap_ci_f1fake']); print("McNemar:",RESULTS['mcnemar_logreg_vs_svm'])


## 9. Assemble results and summary tables

In [ ]:
with open("results/results.json","w") as fp: json.dump(RESULTS, fp, indent=2)

def row(name, r): return f"{name:<26} acc={r['accuracy']:.3f}  F1m={r['f1_macro']:.3f}  AUC={r['roc_auc']:.3f}"
print("="*64,"\nIN-DOMAIN\n"+"="*64)
for dn in DS:
    print(f"\n[{dn}]")
    for fam,key in [("classical","classical_in_domain"),("bilstm","bilstm_in_domain"),
                    ("transformer","transformer_in_domain")]:
        blk=RESULTS[key]
        if fam=="classical":
            for m,r in blk[dn].items(): print(" ",row(m,r))
        elif fam=="bilstm":
            print(" ",row("BiLSTM",blk[dn]))
        else:
            for m in blk: print(" ",row(m,blk[m][dn]))
print("\n"+"="*64,"\nCROSS-DOMAIN (train -> test)\n"+"="*64)
for k,r in RESULTS['classical_cross_domain'].items(): print("  LogReg   ",row(k,r))
for k,r in RESULTS['bilstm_cross_domain'].items():   print("  BiLSTM   ",row(k,r))
for m in RESULTS['transformer_cross_domain']:
    for k,r in RESULTS['transformer_cross_domain'][m].items(): print(f"  {m:<9}",row(k,r))
print("\nResults written to results/results.json")
print("Download it and paste back so the paper's tables are populated from real runs.")
